# 02 Spectral embeddings

Use Laplacian-based spectral embedding to assign dimensionality to graph

## preamble

In [2]:
import numpy as np
import polars as pl
import altair as alt

import pawncounter

In [3]:
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [4]:
BOARD_WIDTH = 2
BOARD_DEPTH = 3

assert 0 <= BOARD_WIDTH <= 8
assert 0 <= BOARD_DEPTH <= 8

pc = pawncounter.PawnCounter(BOARD_WIDTH, BOARD_DEPTH)

In [5]:
transitions = pc.generate_transitions()
positions = pc.extract_positions(transitions)

## calculation

In [6]:
spectral_embeddings = pc.spectral_embedding_3d(
    transitions,
    positions,
    weight=pl.col("transition_type").replace_strict(
        {
            "A": 1 / 1.0,
            "CK": 1 / 1.0,
            "CQ": 1 / 1.0,
            "R": 1 / 1.0,
        }
    ),
)
spectral_embeddings

/home/nic/repos/movpasd/pawncounter/.venv/lib/python3.14/site-packages/sklearn/manifold/_spectral_embedding.py:430: UserWarning: Exited at iteration 20 with accuracies 
[1.69210791e-16 1.69377807e-06 6.76736164e-07 1.45837690e-06
 2.50103466e-04]
not reaching the requested tolerance 4.187226295471191e-06.
Use iteration 19 instead with accuracy 
4.3468823185094686e-05.

  _, diffusion_map = lobpcg(laplacian, X, M=M, tol=tol, largest=False)
/home/nic/repos/movpasd/pawncounter/.venv/lib/python3.14/site-packages/sklearn/manifold/_spectral_embedding.py:430: UserWarning: Exited postprocessing with accuracies 
[2.39474187e-16 1.69403112e-06 6.83474448e-07 1.45739580e-06
 2.13509215e-04]
not reaching the requested tolerance 4.187226295471191e-06.
  _, diffusion_map = lobpcg(laplacian, X, M=M, tol=tol, largest=False)


position_id,pos,x,y,z
u32,u128,f64,f64,f64
0,18963252907773419061505,-9.3711e-8,-1.9393e-8,-0.039049
1,18963252907773419061506,-0.018115,0.003353,-0.030406
2,18963252907773419062016,-0.000752,0.013993,-0.029499
3,18963252907773419061504,-0.008202,0.007456,-0.033412
4,18963252907773419061761,0.018115,-0.003353,-0.030406
…,…,…,…,…
276,4740813226943354766338,-0.014675,-0.012466,0.038346
277,4740813226943354766848,-0.005064,-0.009999,0.038992
278,4759259971017064317956,0.014675,-0.012466,0.038346


## altair-based 2D visualisation

In [7]:
_DEPTH = 3

_TRANSITION_COLOURS = {
    "domain": ["A", "CK", "CQ", "R"],
    "range": ["firebrick", "green", "aqua", "grey"],
}

_edges_df = (
    transitions.lazy()
    .filter(pl.col("transition_depth") < _DEPTH)
    .join(
        spectral_embeddings.lazy().select(start_pos="pos", x0="x", y0="y", z0="z"),
        on="start_pos",
    )
    .join(
        spectral_embeddings.lazy().select(end_pos="pos", x1="x", y1="y", z1="z"),
        on="end_pos",
    )
    .collect()
)

(
    alt.Chart(_edges_df)
    .mark_rule(strokeWidth=5, opacity=0.3)
    .encode(
        alt.X("x0"),
        alt.X2("x1"),
        alt.Y("y0"),
        alt.Y2("y1"),
        alt.Color("transition_type:N").scale(**_TRANSITION_COLOURS),  # type: ignore
    )
    .properties(width=300, height=300)
)

alt.Chart(...)

## 3D render

In [8]:
@pc.button("Render transition mesh")
def render_transition_mesh():
    # Line mesh of transitions near the start, coloured by transition_type. Like the
    # Altair cell, only keep transitions whose source is within _DEPTH BFS levels.
    _DEPTH = 3

    # pos -> global position_id, and the embedding coordinates (row k == position_id k).
    _id = spectral_embeddings.select("pos", "position_id")
    _emb = spectral_embeddings.sort("position_id").select("x", "y", "z").to_numpy()

    # Edges: depth-filtered transitions as global position_id index pairs.
    _edges_df = (
        transitions.lazy()
        .filter(pl.col("transition_depth") < _DEPTH)
        .join(
            _id.lazy().rename({"pos": "start_pos", "position_id": "i"}), on="start_pos"
        )
        .join(_id.lazy().rename({"pos": "end_pos", "position_id": "j"}), on="end_pos")
        .collect()
    )
    _edges = _edges_df.select("i", "j").to_numpy()

    _TRANSITION_COLOURS = {
        "A": (0.70, 0.13, 0.13),  # firebrick
        "CK": (0.00, 0.50, 0.00),  # green
        "CQ": (0.00, 1.00, 1.00),  # aqua
        "R": (0.50, 0.50, 0.50),  # grey
    }
    _edge_colours = np.array(
        [_TRANSITION_COLOURS[t] for t in _edges_df["transition_type"].cast(pl.String)]
    )

    # The initial state (BFS root) is the start_pos of the depth-0 transitions.
    _initial_pos = pc.init_position()
    _initial_pos_int = pc.position_to_int(_initial_pos)
    _initial_id = spectral_embeddings.filter(pl.col("pos") == _initial_pos_int)[
        "position_id"
    ].item()

    pc.display_coloured_mesh(
        _emb,
        _edges,
        _edge_colours,
        name="transitions",
        highlight_nodes=[_initial_id],  # type: ignore
    )


render_transition_mesh()


Button(description='Render transition mesh', style=ButtonStyle())